*Writing code to build a **XG Boost model** to predict malignant/benign*

In [28]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV,
)
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, classification_report

from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

In [29]:
RANDOM_STATE=42
CV_FOLDS=5
TEST_SIZE=0.2

PARAM_GRID = {
    "model__n_estimators"     : [100, 200],
    "model__max_depth"        : [3, 6, 9],
    "model__learning_rate"    : [0.01, 0.1],
    "model__subsample"        : [0.8, 1.0],
    "model__colsample_bytree" : [0.8, 1.0],
}

In [35]:
def load_data():
    data=load_breast_cancer()
    X=data.data
    feature_names = data.feature_names
    class_names = data.target_names
    print(f"shape:{X.shape}\nfeature names:{list(feature_names)}")
    y=data.target
    print(f"shape:{y.shape}\ntarget names:{class_names}")
    return X, y, feature_names, class_names

def split_data(X, y):
    X_train, X_test, y_train, y_test=train_test_split(
       X, y,
       test_size=TEST_SIZE,
       random_state=RANDOM_STATE,
       stratify=y,
    )
    print(f"\ntrain size:{X_train.shape[0]}")
    print(f"test size:{X_test.shape[0]}")
    return X_train, X_test, y_train, y_test

In [31]:
def the_pipeline():
    return Pipeline([
           ("scaler", StandardScaler()),
           ("model", XGBClassifier(random_state=RANDOM_STATE, n_jobs=-1, eval_metric="logloss")),
])

def cross_validation(pipeline, X, y):
    scoring=cross_val_score(
        pipeline, X, y,
        cv=CV_FOLDS,
        scoring="accuracy",
        n_jobs=-1,
    )
    print(f"\ncv mean:{scoring.mean():.2f} \ncv std:{scoring.std():.2f}")
    return scoring

In [32]:
#fine tuning if needed to get the best model
def model_tuning(pipeline, X_train, y_train):
    grid_search=GridSearchCV(
        estimator=pipeline,
        param_grid=PARAM_GRID,
        cv=CV_FOLDS,
        n_jobs=-1,
        scoring="accuracy",
        verbose=1,
    )
    grid_search.fit(X_train, y_train)
    print(f"\nthe winning parameter combo:{grid_search.best_params_}")
    print(f"best CV score:{grid_search.best_score_:.3f}")
    return grid_search


In [33]:
def evaluate_model(model, X_test, y_test, class_names):
    y_pred=model.predict(X_test)
    accuracy=accuracy_score(y_test, y_pred)
    print(f"\ntest accuracy : {accuracy:.3f}")
    print(f"\n{classification_report(y_test, y_pred, target_names=class_names)}")
    return y_pred

def print_feature_importance(model, feature_names, top_n=10):
    xgb_step    = model.named_steps["model"]
    importances = xgb_step.feature_importances_
    indices     = np.argsort(importances)[::-1]

    for rank, idx in enumerate(indices[:top_n]):
        bar = "█" * int(importances[idx] * 200)
        print(f"  {rank+1:2}. {feature_names[idx]:35} {importances[idx]:.4f}  {bar}")

In [36]:
def main():
    X, y, feature_names, class_names=load_data()
    X_train, X_test, y_train, y_test=split_data(X, y)

    pipeline=the_pipeline()
    cross_validation(pipeline, X, y)

    pipeline=the_pipeline()
    grid_search=model_tuning(pipeline, X_train, y_train)
    best_model=grid_search.best_estimator_

    evaluate_model(best_model, X_test, y_test, class_names)
    print_feature_importance(best_model, feature_names)

if __name__ == "__main__":
    main()

shape:(569, 30)
feature names:[np.str_('mean radius'), np.str_('mean texture'), np.str_('mean perimeter'), np.str_('mean area'), np.str_('mean smoothness'), np.str_('mean compactness'), np.str_('mean concavity'), np.str_('mean concave points'), np.str_('mean symmetry'), np.str_('mean fractal dimension'), np.str_('radius error'), np.str_('texture error'), np.str_('perimeter error'), np.str_('area error'), np.str_('smoothness error'), np.str_('compactness error'), np.str_('concavity error'), np.str_('concave points error'), np.str_('symmetry error'), np.str_('fractal dimension error'), np.str_('worst radius'), np.str_('worst texture'), np.str_('worst perimeter'), np.str_('worst area'), np.str_('worst smoothness'), np.str_('worst compactness'), np.str_('worst concavity'), np.str_('worst concave points'), np.str_('worst symmetry'), np.str_('worst fractal dimension')]
shape:(569,)
target names:['malignant' 'benign']

train size:455
test size:114

cv mean:0.97 
cv std:0.01
Fitting 5 folds fo